# D4-02 AWARE and biodiversity with regionalised methods

This notebook compares two real BAFU tomato datasets with the same functional unit of **1 kg fresh tomatoes**:
- **`CH early`**: Switzerland, heated greenhouse, early harvest
- **`ES greenhouse`**: Spain, greenhouse production

The goal is to see how the same product family can look very different once regionalized methods respond to **where water evaporation happens** and **where land-related pressures happen**.



## Learning goals

After this notebook, you should be able to:
- run two "real" `edges` methods on two comparable tomato datasets
- compare scores **within** one impact category across `CH early` and `ES greenhouse`
- connect large AWARE differences to water-related exchanges and large IBIF differences to land-use-related exchanges
- reuse an existing `EdgeLCIA` object with `redo_lcia()` instead of rebuilding everything from scratch
- generate a Sankey-style supply-chain view for one tomato case



## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087-3101. https://doi.org/10.1007/s11367-025-02551-7
- Seitfudem, G., Berger, M., Schmied, H. M., & Boulay, A. (2025). The updated and improved method for water scarcity impact assessment in LCA, AWARE2.0. *Journal of Industrial Ecology, 29*(3), 891-907. https://doi.org/10.1111/jiec.70023
- Schipper, A. M., et al. (2025). Intactness-based biodiversity impact factors for life cycle assessment. *Scientific Data*. https://doi.org/10.1038/s41597-025-05946-1



## 1) Select two comparable tomato datasets

We keep the product family fixed and compare two tomato datasets that differ by geography and production system: one Swiss heated-greenhouse system (`CH early`) and one Spanish greenhouse dataset (`ES greenhouse`).


In [ ]:
import io
import time
from contextlib import redirect_stderr, redirect_stdout
from pathlib import Path
from textwrap import fill

import country_converter as coco
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from matplotlib.lines import Line2D

import bw2data as bd
from edges import EdgeLCIA
from edges.supply_chain import SupplyChain

pd.set_option('display.max_colwidth', None)
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size': 8,
})

In [ ]:
bd.projects.set_current('paris-lca-course-2026')
db = bd.Database('bafu')

In [ ]:
list(bd.databases)

In [ ]:
selected_datasets = []
for ds in db:
    is_ch_early = (
        ds["name"] == "Tomatoes conventional, hors-sol heated, at greenhouse, early harvest"
        and ds["location"] == "CH"
    )
    is_es_greenhouse = (
        ds["name"] == "Tomatoes conventional, hors-sol, at greenhouse"
        and ds["location"] == "ES"
    )
    if is_ch_early or is_es_greenhouse:
        selected_datasets.append(ds)

dataset_labels = {
    ("Tomatoes conventional, hors-sol heated, at greenhouse, early harvest", "CH"): "CH early",
    ("Tomatoes conventional, hors-sol, at greenhouse", "ES"): "ES greenhouse",
}

desired_order = ["CH early", "ES greenhouse"]
datasets = []
for desired_label in desired_order:
    for dataset in selected_datasets:
        dataset_key = (dataset["name"], dataset["location"])
        dataset_label = dataset_labels[dataset_key]
        if dataset_label == desired_label:
            datasets.append(dataset)
            break

dataset_order = []
tomato_activities = {}
dataset_rows = []
for dataset in datasets:
    dataset_key = (dataset["name"], dataset["location"])
    dataset_label = dataset_labels[dataset_key]

    dataset_order.append(dataset_label)
    tomato_activities[dataset_label] = dataset

    row = {
        "dataset label": dataset_label,
        "name": dataset["name"],
        "reference product": dataset["reference product"],
        "location": dataset["location"],
        "unit": dataset["unit"],
    }
    dataset_rows.append(row)

pd.DataFrame(dataset_rows)



## 2) Define two real `edges` methods

We will compare:
- `AWARE 2.0`: a regionalized water-scarcity method that is sensitive to where water use and evaporation occur
- `IBIF biodiversity (land use)`: a regionalized biodiversity method focused on **land occupation** and **land transformation** pressures

This combination is useful for tomatoes because one category is strongly water-driven, while the other reacts much more to land-related exchanges.


In [ ]:
aware_method = ('AWARE 2.0', 'Country', 'all', 'yearly')
ibif_landuse_method = ('IBIF', 'biodiversity', 'LU', 'overall')

In [ ]:
# we can check those methods do exist
from edges import get_available_methods

available_methods = get_available_methods()
for method in (aware_method, ibif_landuse_method):
    assert method in available_methods



## 3) Run the full `edges` workflow for each dataset and category

The helper function below keeps the repeated `edges` steps together so the notebook stays readable. We stay explicit here because the mapping sequence is part of the Day 2 story.


In [ ]:
datasets

In [ ]:
category_methods = {
    "AWARE 2.0 water scarcity": aware_method,
    "IBIF biodiversity (land use)": ibif_landuse_method,
}
category_colors = {
    "AWARE 2.0 water scarcity": "#1f77b4",
    "IBIF biodiversity (land use)": "#2e8b57",
}
method_units = {
    "AWARE 2.0 water scarcity": "m$^3$",
    "IBIF biodiversity (land use)": "DALY",
}


def build_split_details(lca):
    rows = []
    for cf in lca.scenario_cfs:
        reporting_split = cf.get("reporting_split")
        if not reporting_split:
            continue

        original_location = cf.get("consumer", {}).get("location")
        for component in reporting_split:
            row = {
                "original consumer location": original_location,
                "split consumer location": component["consumer_location"],
                "weight used": component.get("weight"),
                "share used in split": component["share"],
                "component CF": component["value"],
            }
            rows.append(row)

    split_details = pd.DataFrame(rows)
    if split_details.empty:
        return split_details

    split_details = split_details.sort_values(
        ["original consumer location", "split consumer location", "share used in split"],
        ascending=[True, True, False],
    )
    split_details = split_details.reset_index(drop=True)
    return split_details


results = {}

for category_label, method in category_methods.items():
    results[category_label] = {}

    for i, dataset in enumerate(datasets):
        dataset_label = dataset_labels[(dataset["name"], dataset["location"])]

        if i == 0:
            lca = EdgeLCIA({dataset: 1}, method)
            lca.lci()
            lca.map_exchanges()
            lca.map_aggregate_locations()
            lca.map_dynamic_locations()
            lca.map_contained_locations()
            lca.map_remaining_locations_to_global()
            lca.evaluate_cfs()
            lca.lcia()
        else:
            lca.redo_lcia({dataset: 1})

        matched = lca.generate_cf_table()
        matched_split = lca.generate_cf_table(split_aggregate_consumers=True)
        split_details = build_split_details(lca)
        weighting_metadata = lca.method_metadata.get("weighting_metadata", {})

        result = {
            "score": lca.score,
            "matched": matched,
            "matched_split": matched_split,
            "split_details": split_details,
            "weighting_metadata": weighting_metadata,
        }
        results[category_label][dataset_label] = result


In [ ]:
def dominant_value(df, column):
    if df.empty or column not in df.columns:
        return None

    values = df[column].dropna()
    if values.empty:
        return None

    return values.value_counts().idxmax()


summary_rows = []
for category_label, dataset_results in results.items():
    for dataset_label, result in dataset_results.items():
        row = {
            "dataset label": dataset_label,
            "dataset location": tomato_activities[dataset_label].get("location"),
            "category": category_label,
            "score": result["score"],
            "matched CF rows": len(result["matched"]),
            "dominant supplier flow": dominant_value(result["matched"], "supplier name"),
            "dominant consumer location": dominant_value(result["matched"], "consumer location"),
        }
        summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary = summary.reset_index(drop=True)
summary["dataset label"] = pd.Categorical(
    summary["dataset label"],
    categories=dataset_order,
    ordered=True,
)
summary["category"] = pd.Categorical(
    summary["category"],
    categories=list(category_methods),
    ordered=True,
)
summary = summary.sort_values(["category", "dataset label"])
summary = summary.reset_index(drop=True)
summary


### Optional: compare impact distributions before and after splitting aggregate consumer rows

In these real BAFU tomato cases, `edges` can build weighted-average CF rows for aggregate or dynamic consumer locations such as `RER` or `RoW`. In deterministic mode, `generate_cf_table(split_aggregate_consumers=True)` replaces those rows with country rows using the exact shares stored in `scenario_cfs[*]["reporting_split"]`.

The short example below uses the first AWARE tomato case (`CH early`) and compares the impact distribution by consumer location before and after splitting. It also shows the raw weights and normalized shares used for the split.


In [ ]:
def split_demo_impact_column(df):
    if "impact (mean)" in df.columns:
        return "impact (mean)"
    return "impact"


def plot_split_comparison(result, dataset_label, category_label, top_n=10):
    unsplit = result["matched"]
    split = result["matched_split"]
    impact_unsplit = split_demo_impact_column(unsplit)
    impact_split = split_demo_impact_column(split)

    unsplit_by_location = unsplit.groupby("consumer location")[impact_unsplit].sum()
    split_by_location = split.groupby("consumer location")[impact_split].sum()

    comparison = pd.DataFrame()
    comparison["without split"] = unsplit_by_location
    comparison["with split"] = split_by_location
    comparison = comparison.fillna(0.0)

    max_abs_values = comparison.abs().max(axis=1)
    top_locations = max_abs_values.sort_values(ascending=False).head(top_n).index
    comparison = comparison.loc[top_locations]

    ax = comparison.plot(
        kind="bar",
        figsize=(9, 4),
        color=["#9ecae1", "#1f77b4"],
        width=0.85,
    )
    ax.set_title(f"{category_label} | {dataset_label}: impact by consumer location")
    ax.set_ylabel(method_units[category_label])
    ax.set_xlabel("Consumer location")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

    return comparison


aware_split_demo = dataset_order[0]
aware_result = results["AWARE 2.0 water scarcity"][aware_split_demo]
weight_source = aware_result["weighting_metadata"].get("label", "unknown")

unsplit_total = aware_result["matched"][
    split_demo_impact_column(aware_result["matched"])
].sum()
split_total = aware_result["matched_split"][
    split_demo_impact_column(aware_result["matched_split"])
].sum()
totals_match = np.isclose(unsplit_total, split_total)

print("AWARE tomato dataset used for split demo:", aware_split_demo)
print("Effective split weight source:", weight_source)
print("Total impact preserved:", totals_match)

comparison = plot_split_comparison(
    aware_result,
    dataset_label=aware_split_demo,
    category_label="AWARE 2.0 water scarcity",
)
display(comparison)

split_preview_columns = [
    "original consumer location",
    "split consumer location",
    "weight used",
    "share used in split",
]
split_preview = aware_result["split_details"][split_preview_columns].head(12)
display(split_preview)


## 4) Build a four-panel figure

The `edges` AWARE example notebook uses a compact four-panel layout to connect locations, CFs, process contributions, and final scores. Below we adapt that plotting pattern to the tomato case and show the **highest-scoring tomato dataset in each category**.

For consistency with the split demo and the later choropleth maps, the consumer-location panels below use the split reporting table.


In [ ]:
def impact_column(df):
    if "impact (mean)" in df.columns:
        return "impact (mean)"
    return "impact"


def cf_column(df):
    if "CF (mean)" in df.columns:
        return "CF (mean)"
    return "CF"


def plot_publication_style(result, dataset_label, category_label, top_n=5):
    df = result["matched_split"].copy()
    if df.empty:
        raise ValueError(f"No matched CF rows for {dataset_label} / {category_label}")

    color = category_colors[category_label]
    unit = method_units[category_label]
    score_series = score_table[category_label].loc[dataset_order]
    impact_col = impact_column(df)
    cf_col = cf_column(df)

    df_positive = df[df[cf_col] > 0].copy()
    if df_positive.empty:
        df_positive = df.copy()

    fig, axes = plt.subplots(2, 2, figsize=(8.6, 6.6))
    axes = axes.flatten()

    if category_label == "AWARE 2.0 water scarcity":
        df_net = df.copy()
        df_net.loc[df_net[cf_col] < 0, "amount"] *= -1
        location_values = df_net.groupby("consumer location")["amount"].sum()
        location_values = location_values.sort_values(ascending=False)
        location_values = location_values[location_values > 0]
        location_values = location_values.head(top_n)

        axes[0].set_title("a) Net freshwater use")
        axes[0].set_ylabel("m$^3$")
        if len(location_values) > 1 and location_values.iloc[-1] > 0:
            ratio = location_values.iloc[0] / location_values.iloc[-1]
            if ratio > 30:
                axes[0].set_yscale("log")
    else:
        location_values = df_positive.groupby("consumer location")[impact_col].sum()
        location_values = location_values.sort_values(ascending=False)
        location_values = location_values.head(top_n)
        axes[0].set_title("a) Impact by consumer location")
        axes[0].set_ylabel(unit)

    axes[0].bar(location_values.index, location_values.values, color=color, width=0.6)
    axes[0].set_xticks(range(len(location_values)))
    axes[0].set_xticklabels(location_values.index, rotation=0)

    location_cfs = df_positive.groupby("consumer location")[cf_col].mean()
    location_cfs = location_cfs.reindex(location_values.index)
    location_cfs = location_cfs.fillna(0)
    axes[1].bar(location_cfs.index, location_cfs.values, color=color, width=0.6)
    axes[1].set_title("b) Mean positive characterization factor")
    axes[1].set_ylabel("CF")
    axes[1].set_xticks(range(len(location_cfs)))
    axes[1].set_xticklabels(location_cfs.index, rotation=0)

    plot_df = df_positive.copy()
    short_names = plot_df["consumer name"].fillna("Unknown").str.split(", ").str[0]
    short_locations = plot_df["consumer location"].fillna("n/a")
    plot_df["label"] = short_names + " (" + short_locations + ")"
    process_impacts = plot_df.groupby("label")[impact_col].sum()
    process_impacts = process_impacts.sort_values(ascending=False)
    process_impacts = process_impacts.head(top_n)

    x = np.arange(len(process_impacts))
    axes[2].bar(x, process_impacts.values, color=color, width=0.6)
    axes[2].set_title("c) Process contribution (positive CFs)")
    axes[2].set_ylabel(unit)

    wrapped_names = []
    process_locations = []
    for label in process_impacts.index:
        if label.endswith(")") and " (" in label:
            process_name, process_location = label.rsplit(" (", 1)
            process_location = process_location[:-1]
        else:
            process_name = label
            process_location = ""

        wrapped_names.append(fill(process_name, 16))
        process_locations.append(process_location)

    axes[2].set_xticks(x)
    axes[2].set_xticklabels(wrapped_names, rotation=0, ha="center", size=7)
    for xi, yi, code in zip(x, process_impacts.values, process_locations):
        axes[2].annotate(
            code,
            (xi, yi),
            textcoords="offset points",
            xytext=(0, 3),
            ha="center",
            fontsize=7,
        )

    highlight_colors = []
    for label in dataset_order:
        if label == dataset_label:
            highlight_colors.append(color)
        else:
            highlight_colors.append("#D9D9D9")

    axes[3].bar(dataset_order, score_series.values, color=highlight_colors, width=0.6)
    axes[3].set_title("d) LCIA score across tomato datasets")
    axes[3].set_ylabel(unit)
    axes[3].legend(
        handles=[
            Line2D([0], [0], color=color, lw=6, label="Highlighted dataset"),
            Line2D([0], [0], color="#D9D9D9", lw=6, label="Other tomato datasets"),
        ],
        loc="upper right",
        fontsize=7,
    )

    fig.suptitle(f"{category_label} | {dataset_label}", y=0.98, fontsize=9)
    plt.tight_layout(rect=[0, 0.02, 1, 0.95])
    plt.subplots_adjust(hspace=0.45, wspace=0.28)
    plt.show()


In [ ]:
score_table_data = summary[["dataset label", "category", "score"]]
score_table = score_table_data.pivot(
    index="dataset label",
    columns="category",
    values="score",
)
score_table = score_table.loc[dataset_order]

score_table

plot_targets = {}
for category_label in category_methods:
    plot_targets[category_label] = score_table[category_label].idxmax()

for category_label, dataset_label in plot_targets.items():
    plot_publication_style(
        results[category_label][dataset_label],
        dataset_label=dataset_label,
        category_label=category_label,
    )


## 5) Put the location pattern on a map

The map complements the score table and the publication-style plots: it shows **where the characterized exchanges actually occur**, not just how large the total score is.

Because the choropleth is country-based, we use the split CF tables below. This expands weighted fallback consumer rows such as `RER` or `RoW` into country rows with the stored shares, while preserving the total impact for each tomato dataset.

For **AWARE**, we map **net** location impacts so the map stays aligned with the final score. This matters because positive and negative water rows can offset each other strongly. For **IBIF land use**, we keep mapping **absolute** contributions because the goal is to show where land-related burdens are distributed across the supply chain.


In [ ]:
location_share_modes = {
    "AWARE 2.0 water scarcity": "net",
    "IBIF biodiversity (land use)": "absolute",
}


def location_shares(result, mode):
    df = result["matched_split"].copy()
    impact_col = impact_column(df)

    if mode == "net":
        contribution_col = impact_col
        contribution_label = "net characterized contribution"
    else:
        contribution_col = "absolute contribution"
        contribution_label = "absolute characterized contribution"
        df[contribution_col] = df[impact_col].abs()

    shares = df.groupby("consumer location")[contribution_col].sum()
    shares = shares.sort_values(ascending=False)
    shares = shares.reset_index()
    shares = shares.rename(columns={contribution_col: contribution_label})

    total_contribution = shares[contribution_label].sum()
    shares["share"] = 0.0
    if total_contribution != 0:
        shares["share"] = shares[contribution_label] / total_contribution
    shares["share_pct"] = 100 * shares["share"]
    shares["mode"] = mode

    return shares


def to_iso3(location):
    sink = io.StringIO()
    with redirect_stdout(sink), redirect_stderr(sink):
        iso3 = coco.convert(names=location, to="ISO3", not_found=None)
    if not iso3 or iso3 == location or len(str(iso3)) != 3:
        return None
    return str(iso3)


def plot_location_share_map(location_df, title, color_scale, midpoint=None, range_color=None):
    rows = []
    skipped = []

    for _, row in location_df.iterrows():
        iso3 = to_iso3(row["consumer location"])
        if iso3 is None:
            skipped.append(row["consumer location"])
            continue

        map_row = {
            "location": row["consumer location"],
            "iso3": iso3,
            "share_pct": row["share_pct"],
        }
        rows.append(map_row)

    frame = pd.DataFrame(rows)
    fig = px.choropleth(
        frame,
        locations="iso3",
        color="share_pct",
        hover_name="location",
        projection="natural earth",
        color_continuous_scale=color_scale,
        title=title,
        color_continuous_midpoint=midpoint,
        range_color=range_color,
    )
    fig.update_geos(showcoastlines=True, showcountries=True, fitbounds="locations")
    fig.update_layout(
        width=1200,
        height=800,
        margin=dict(l=20, r=20, t=80, b=20),
    )
    fig.show()

    if skipped:
        print("Not shown on the country-level map:", skipped)


location_share_tables = {}
for dataset_label in dataset_order:
    location_share_tables[dataset_label] = {}
    for category_label in category_methods:
        result = results[category_label][dataset_label]
        mode = location_share_modes[category_label]
        location_share_tables[dataset_label][category_label] = location_shares(result, mode)

for category_label in category_methods:
    print(f"Top location shares for {category_label}")

    display_frames = []
    for dataset_label in dataset_order:
        location_df = location_share_tables[dataset_label][category_label]
        preview = location_df.head(6).copy()
        preview["dataset label"] = dataset_label
        preview["dataset location"] = tomato_activities[dataset_label].get("location")
        preview["weight source"] = results[category_label][dataset_label]["weighting_metadata"].get("label", "unknown")
        preview["table used"] = "matched_split"
        preview["map basis"] = location_share_modes[category_label]
        display_frames.append(preview)

    display_frame = pd.concat(display_frames, ignore_index=True)
    display(display_frame)


In [ ]:
for dataset_label in dataset_order:
    aware_map = location_share_tables[dataset_label]['AWARE 2.0 water scarcity']
    aware_max = aware_map['share_pct'].max()
    plot_location_share_map(
        aware_map,
        f'AWARE 2.0 ({dataset_label} tomatoes): share of total net characterized impact by consumer location',
        [(0.0, "white"), (1.0, "darkred")],
        range_color=(0, aware_max),
    )

for dataset_label in dataset_order:
    plot_location_share_map(
        location_share_tables[dataset_label]['IBIF biodiversity (land use)'],
        f'IBIF land use ({dataset_label} tomatoes): share of total absolute characterized impact by consumer location',
        'Greens',
    )


### Reading the maps

A few patterns are worth making explicit:

- For **AWARE**, the maps now show **net** location impacts with a scale that runs from **white at 0** to **dark red at the largest positive net contribution**. In the `ES greenhouse` case, **Spain** dominates the final water-scarcity score because the main consumptive water use is local. Countries such as **Italy**, **Greece**, or **Switzerland** can still host large water-related exchanges, especially hydropower-related rows, but many positive and negative contributions offset each other strongly, so they do not necessarily appear as dark hotspots on the net map.
- For **IBIF land use**, much of the land-use burden sits in the **upstream supply chain**, especially forestry, crop, and material inputs located in countries such as **Sweden** and **Germany**.
- This is why the `CH early` tomato still shows large **SE** and **DE** shares under IBIF, and why the `ES greenhouse` tomato does **not** show Spain as a major land-use hotspot: the Spanish greenhouse dataset has very little characterized land occupation in `ES` itself, while many upstream land-use burdens occur elsewhere.


## 6) Generate a Sankey diagram

`SupplyChain` gives a complementary view of the score distribution in the technosphere. Internally, it reuses one `EdgeLCIA` object and repeatedly calls `redo_lcia()` while traversing the graph.

Below we build a Sankey for the **Spain tomato dataset** under **IBIF biodiversity (land use)**, using a **0.1% cutoff** of the total score.

In [ ]:
sankey_dataset_label = "ES greenhouse"
sankey_activity = tomato_activities[sankey_dataset_label]

ibif_supply_chain = SupplyChain(
    activity=sankey_activity,
    method=ibif_landuse_method,
    amount=1,
    level=5,
    cutoff=0.01,
    cutoff_basis="total",
)

sankey_total_score = ibif_supply_chain.bootstrap()
sankey_df, sankey_total_score, sankey_reference_amount = ibif_supply_chain.calculate()

display(
    sankey_df[["level", "name", "location", "amount", "score", "share_of_total"]].head(12)
)
print("Sankey root dataset:", sankey_dataset_label)
print("Reference amount:", sankey_reference_amount)
print("Total IBIF score:", sankey_total_score)

In [ ]:
sankey_path = Path.cwd() / "D4-02_ibif_es_tomatoes_sankey.html"

sankey_fig = ibif_supply_chain.plot_sankey(
    sankey_df,
    width_max=1600,
    height_max=700,
    auto_width=True,
)

ibif_supply_chain.save_html(
    sankey_df,
    str(sankey_path),
    width_max=1600,
    height_max=700,
    auto_width=True,
)

print(f"Saved Sankey HTML to: {sankey_path}")
sankey_fig

## 7) Reuse one `EdgeLCIA` object with `redo_lcia()`

To make the reuse pattern more visible, we now keep **one method fixed** and switch the demand across five reproducibly sampled BAFU datasets.

Below, `redo_lcia()` is compared against a fresh full rerun for each sampled dataset. The point is not the specific activities, but that the same characterization setup can be reused when only the demand changes.

In [ ]:
redo_method = aware_method
redo_sample_size = 5

excluded_activity_keys = set()
for activity in tomato_activities.values():
    excluded_activity_keys.add((activity["database"], activity["code"]))

redo_pool = []
for ds in db:
    activity_key = (ds["database"], ds["code"])
    if activity_key in excluded_activity_keys:
        continue
    if not ds.get("location"):
        continue
    redo_pool.append(ds)

redo_pool = sorted(
    redo_pool,
    key=lambda ds: (
        ds["name"],
        ds.get("location") or "",
        ds.get("reference product") or "",
    ),
)

rng = np.random.default_rng(42)
sampled_indices = rng.choice(len(redo_pool), size=redo_sample_size, replace=False)
redo_activities = []
for index in sampled_indices:
    redo_activities.append(redo_pool[index])

redo_sample_rows = []
for i, activity in enumerate(redo_activities, start=1):
    row = {
        "sample": i,
        "name": activity["name"],
        "reference product": activity.get("reference product"),
        "location": activity.get("location"),
        "unit": activity.get("unit"),
    }
    redo_sample_rows.append(row)

redo_sample = pd.DataFrame(redo_sample_rows)
display(redo_sample)

redo_demo = EdgeLCIA({redo_activities[0]: 1}, redo_method)
redo_demo.lci()
redo_demo.apply_strategies()
redo_demo.evaluate_cfs()
redo_demo.lcia()

redo_rows = []
for i, activity in enumerate(redo_activities, start=1):
    t0 = time.perf_counter()
    redo_demo.redo_lcia({activity: 1}, recompute_score=True)
    redo_elapsed = time.perf_counter() - t0

    fresh = EdgeLCIA({activity: 1}, redo_method)
    t1 = time.perf_counter()
    fresh.lci()
    fresh.apply_strategies()
    fresh.evaluate_cfs()
    fresh.lcia()
    fresh_elapsed = time.perf_counter() - t1

    row = {
        "sample": i,
        "name": activity["name"],
        "location": activity.get("location"),
        "redo_lcia score": redo_demo.score,
        "fresh rerun score": fresh.score,
        "absolute difference": abs(redo_demo.score - fresh.score),
        "redo seconds": redo_elapsed,
        "fresh rerun seconds": fresh_elapsed,
    }
    redo_rows.append(row)

redo_table = pd.DataFrame(redo_rows)
redo_table


## Checkpoint 1

- Which tomato dataset has the largest AWARE score? Which has the smallest?
- Which tomato dataset has the largest IBIF land-use score? Which has the smallest?
- For AWARE, why do positive and negative water contributions offset each other?
- What does the Sankey add that the score table alone does not?

## Recap

After this notebook, you should now be able to:
- compare two tomato datasets with the same functional unit inside two regionalized impact categories
- explain why AWARE reacts to where water-related exchanges occur and why IBIF land use reacts to where land occupation and transformation occur
- read a compact publication-style `edges` figure, a map-based location-share view, and a Sankey supply-chain breakdown together
- use `generate_cf_table(split_aggregate_consumers=True)` and `reporting_split` to inspect how weighted fallback consumer rows are disaggregated for country-level reporting
- reuse an `EdgeLCIA` object with `redo_lcia()` when the method stays fixed and only the demand changes
